# 02 Results: Final model training and evaluation

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import torch

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

## Best configuration from W&B Sweep

Copy the best hyperparameters from the sweep dashboard and paste them below.

In [4]:
CONFIG = {
    "model_name": "InceptionTime",
    "window_size": 24,
    "horizon": 12,
    "batch_size": 32,          # <- from sweep
    "learning_rate": 0.001,    # <- from sweep
    "epochs": 50,
    "patience": 10,
    "pos_weight": 1.0,         # <- from sweep
    "model_kwargs": {
        "nf": 32,              # <- from sweep
        "depth": 6,            # <- from sweep
        "ks": 40,              # <- from sweep
    },
}

## Load data

In [2]:
from src.data import build_labeled_dataset, make_sliding_windows, normalize_series

DATA_DIR = PROJECT_ROOT / "data"
LABELS_PATH = DATA_DIR / "combined_windows.json"

TRAIN_SERIES = [
    DATA_DIR / "realAWSCloudwatch/ec2_cpu_utilization_24ae8d.csv",
    DATA_DIR / "realAWSCloudwatch/ec2_cpu_utilization_53ea38.csv",
    DATA_DIR / "realAWSCloudwatch/ec2_cpu_utilization_5f5533.csv",
    DATA_DIR / "realAWSCloudwatch/ec2_cpu_utilization_825cc2.csv",
    DATA_DIR / "realAWSCloudwatch/ec2_cpu_utilization_c6585a.csv",
    DATA_DIR / "realAWSCloudwatch/ec2_cpu_utilization_fe7f93.csv",
    DATA_DIR / "realAWSCloudwatch/ec2_disk_write_bytes_1ef3de.csv",
    DATA_DIR / "realAWSCloudwatch/ec2_network_in_257a54.csv",
    DATA_DIR / "realAWSCloudwatch/grok_asg_anomaly.csv",
    DATA_DIR / "realAWSCloudwatch/iio_us-east-1_i-a2eb1cd9_NetworkIn.csv",
    DATA_DIR / "realAWSCloudwatch/rds_cpu_utilization_cc0c53.csv",
]

VAL_SERIES = [
    DATA_DIR / "realAWSCloudwatch/ec2_network_in_5abac7.csv",
    DATA_DIR / "realAWSCloudwatch/elb_request_count_8c0756.csv",
    DATA_DIR / "realAWSCloudwatch/ec2_cpu_utilization_ac20cd.csv",
]

TEST_SERIES = [
    DATA_DIR / "realAWSCloudwatch/ec2_cpu_utilization_77c1ca.csv",
    DATA_DIR / "realAWSCloudwatch/ec2_disk_write_bytes_c0d644.csv",
    DATA_DIR / "realAWSCloudwatch/rds_cpu_utilization_e47b3b.csv",
]

In [5]:
train_df = build_labeled_dataset(TRAIN_SERIES, LABELS_PATH)
train_df = normalize_series(train_df)
X_train, y_train = make_sliding_windows(train_df, CONFIG["window_size"], CONFIG["horizon"])
print(f"Training set: {X_train.shape}, positive ratio: {y_train.mean():.4f}")

val_df = build_labeled_dataset(VAL_SERIES, LABELS_PATH)
val_df = normalize_series(val_df)
X_val, y_val = make_sliding_windows(val_df, CONFIG["window_size"], CONFIG["horizon"])
print(f"Validation set: {X_val.shape}, positive ratio: {y_val.mean():.4f}")

test_df = build_labeled_dataset(TEST_SERIES, LABELS_PATH)
test_df = normalize_series(test_df)
X_test, y_test = make_sliding_windows(test_df, CONFIG["window_size"], CONFIG["horizon"])
print(f"Test set: {X_test.shape}, positive ratio: {y_test.mean():.4f}")

Training set: (42465, 1, 24), positive ratio: 0.0949
Validation set: (12689, 1, 24), positive ratio: 0.1051
Test set: (11991, 1, 24), positive ratio: 0.1064


## Train final model

In [ ]:
import wandb
from src.train import train_model

wandb.init(
    project="Predictive-alerting-for-cloud-metrics",
    entity="tombik-warsaw-university-of-technology",
    name="final-model",
    config=CONFIG,
)

clf = train_model(X_train, y_train, X_val, y_val, CONFIG)

## Save model weights

In [ ]:
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
ARTIFACTS_DIR.mkdir(exist_ok=True)

model_path = ARTIFACTS_DIR / "final_model.pt"
torch.save({"state_dict": clf.state_dict(), "config": CONFIG}, model_path)
print(f"Model saved to {model_path}")

## Find production threshold on validation set

In [ ]:
from src.evaluate import predict_proba, find_optimal_threshold, compute_metrics

val_probs = predict_proba(clf, X_val)
threshold = find_optimal_threshold(y_val, val_probs)
print(f"Production threshold (from validation set): {threshold:.4f}")

## Validation set metrics

In [ ]:
val_metrics = compute_metrics(y_val, val_probs, threshold)
for k, v in val_metrics.items():
    print(f"  val_{k}: {v:.4f}" if isinstance(v, float) else f"  val_{k}: {v}")

## Test set metrics (using validation-derived threshold)

In [ ]:
test_probs = predict_proba(clf, X_test)
test_metrics = compute_metrics(y_test, test_probs, threshold)
for k, v in test_metrics.items():
    print(f"  test_{k}: {v:.4f}" if isinstance(v, float) else f"  test_{k}: {v}")

In [ ]:
wandb.log({f"val_{k}": v for k, v in val_metrics.items()})
wandb.log({f"test_{k}": v for k, v in test_metrics.items()})
wandb.finish()

## Confusion matrix

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ConfusionMatrixDisplay.from_predictions(
    y_val, (val_probs >= threshold).astype(int),
    display_labels=["Normal", "Incident"],
    ax=axes[0],
    cmap="Blues",
)
axes[0].set_title("Validation set")

ConfusionMatrixDisplay.from_predictions(
    y_test, (test_probs >= threshold).astype(int),
    display_labels=["Normal", "Incident"],
    ax=axes[1],
    cmap="Blues",
)
axes[1].set_title("Test set")

plt.tight_layout()
plt.show()

## Precision-Recall curve

In [ ]:
from sklearn.metrics import PrecisionRecallDisplay

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

PrecisionRecallDisplay.from_predictions(y_val, val_probs, ax=axes[0], name="Validation")
axes[0].axvline(val_metrics["recall"], color="r", linestyle="--", label=f"threshold={threshold:.3f}")
axes[0].legend()
axes[0].set_title("Validation set")

PrecisionRecallDisplay.from_predictions(y_test, test_probs, ax=axes[1], name="Test")
axes[1].axvline(test_metrics["recall"], color="r", linestyle="--", label=f"threshold={threshold:.3f}")
axes[1].legend()
axes[1].set_title("Test set")

plt.tight_layout()
plt.show()

## Predictions on raw time series

Overlay model predictions on one of the test series to visualize how alerts would fire.

In [ ]:
# Pick the first test series for visualization
viz_series_path = TEST_SERIES[0]
viz_df = build_labeled_dataset([viz_series_path], LABELS_PATH)
viz_df = normalize_series(viz_df)
X_viz, y_viz = make_sliding_windows(viz_df, CONFIG["window_size"], CONFIG["horizon"])

viz_probs = predict_proba(clf, X_viz)
viz_preds = (viz_probs >= threshold).astype(int)

# Timestamps aligned to the end of each window
timestamps = viz_df.sort_values("timestamp")["timestamp"].values
window_end_timestamps = timestamps[CONFIG["window_size"] - 1 : CONFIG["window_size"] - 1 + len(viz_preds)]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 8), sharex=True)

# Raw time series with incident regions
ax1.plot(viz_df["timestamp"], viz_df["value"], linewidth=0.8, label="value")
incident_mask = viz_df["is_incident"] == 1
ax1.fill_between(
    viz_df["timestamp"], viz_df["value"].min(), viz_df["value"].max(),
    where=incident_mask.values, alpha=0.2, color="red", label="incident window",
)
ax1.set_ylabel("Normalized value")
ax1.legend()
ax1.set_title(viz_series_path.stem)

# Model predictions
ax2.plot(window_end_timestamps, viz_probs, linewidth=0.8, label="P(incident)", color="tab:orange")
ax2.axhline(threshold, color="red", linestyle="--", label=f"threshold={threshold:.3f}")
ax2.fill_between(
    window_end_timestamps, 0, 1,
    where=viz_preds.astype(bool), alpha=0.15, color="red", label="alert",
)
ax2.set_ylabel("Probability")
ax2.set_ylim(-0.05, 1.05)
ax2.legend()

plt.tight_layout()
plt.show()

## Discussion

### Modeling choices
- **InceptionTime** — state-of-the-art deep learning architecture for time-series classification, chosen for its strong performance across diverse benchmarks without heavy feature engineering.
- **Sliding window formulation** — each sample is W time steps of a single metric; the label is whether an incident occurs in the next H steps. This frames the alerting problem as standard binary classification.
- **`pos_weight`** — addresses the heavy class imbalance (incidents are rare) by up-weighting the positive class in the loss function.

### Evaluation setup
- The **production threshold** is selected on the validation set by maximizing F1, then applied unchanged to the test set. This avoids data leakage.
- Metrics reported: accuracy, precision, recall, F1, ROC AUC, and full confusion matrix.

### Limitations
- **Single-metric windows** — the model sees only one metric at a time. Real systems would benefit from multi-variate input.
- **Fixed window/horizon** — W=24 and H=12 are global constants. In production, different services may need different look-back and look-ahead periods.
- **Small dataset** — NAB's realAWSCloudwatch has only 17 series. Results may not generalize to diverse cloud workloads.

### Adapting to a real alerting system
- **Streaming inference** — run the model on a sliding window that advances every time step, emitting an alert when P(incident) exceeds the threshold.
- **Multi-metric input** — extend the window shape from (1, W) to (C, W) where C is the number of correlated metrics (CPU, memory, network, etc.).
- **Threshold tuning per service** — allow operators to adjust the alert threshold based on the cost of false positives vs. missed incidents for each service.
- **Continuous retraining** — periodically retrain on recent data to adapt to workload drift.